# 문법 교정과 번역 기능이 있는 영어 회화 챗봇

지금까지 배운 것(State 확장, 조건부 분기, 구조화 출력, 멀티 노드)을 한데 모은 종합 프로젝트.

동작:
1. 입력이 **한국어**면 → 영어로 **번역(translation)** 후 교정으로
2. 입력이 **영어**면 → 바로 **문법 교정(correction)**
3. 교정 결과를 바탕으로 **답변(respond)** 생성 후 대화 이어가기

```
START ─(한국어?)─> translation ─┐
      └(영어?)──────────────────> correction → respond → END
```

## 환경 변수 준비

이 노트북부터는 실제 LLM 을 호출하므로 API 키가 필요하다.
프로젝트 루트의 `.env` 파일에 키를 넣어두고 `load_dotenv()` 로 불러온다.

```
# .env  (.env.example 을 복사해서 작성)
OPENAI_API_KEY=sk-...
TAVILY_API_KEY=tvly-...   # 웹검색 노트북에서 필요
```

- OpenAI 키: <https://platform.openai.com/api-keys>
- Tavily 키: <https://app.tavily.com>

In [ ]:
import os
from dotenv import load_dotenv

# 프로젝트 루트의 .env 를 찾아 환경변수로 로드
load_dotenv()

# 필요한 키가 있는지 확인
assert os.environ.get("OPENAI_API_KEY"), "OPENAI_API_KEY 가 .env 에 없습니다"
print("환경변수 로드 완료:", "OPENAI_API_KEY")

## Step 1. State 정의
`MessagesState` 를 확장해 교정 관련 필드를 추가한다.

In [ ]:
from langgraph.graph import MessagesState, StateGraph

class State(MessagesState):
    is_correct: bool             # 문장이 이미 올바른지
    corrected_sentence: str      # 교정된 문장
    feedback: str                # 교정 사유 설명

graph_builder = StateGraph(State)

In [ ]:
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(model="gpt-4o")

## Step 2. 번역 노드 (translation)
한국어 입력을 영어로 번역한다.

In [ ]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.messages import AIMessage

TRANSLATION_SYSTEM_TEMPLATE = """
You are a helpful assistant that translates Korean conversation messages into English.
Whenever a Korean message is given, provide its accurate English translation.
Do not wrap the response in any backticks or anything else. Respond with a message only!
"""

TRANSLATION_USER_TEMPLATE = """
Please translate the following Korean message into English:
{messages}
Translation Result:
"""

def translation(state: State):
    translation_msgs = [
        ("system", TRANSLATION_SYSTEM_TEMPLATE),
        ("user", TRANSLATION_USER_TEMPLATE),
    ]
    translation_prompt = ChatPromptTemplate.from_messages(translation_msgs)
    response = llm.invoke(
        translation_prompt.format_messages(messages=state["messages"])
    )
    print("\n[translation node] ->", response.content)
    return {"messages": [AIMessage(content=response.content)]}

graph_builder.add_node("translation", translation)

## Step 3. 문법 교정 노드 (correction)
구조화 출력(`GrammarFeedback`)으로 교정 문장·피드백·정답 여부를 한 번에 받는다.

In [ ]:
from pydantic import BaseModel, Field

class GrammarFeedback(BaseModel):
    """문법 교정 결과"""
    corrected_sentence: str = Field(description="The corrected version of the user's sentence.")
    feedback: str = Field(description="An explanation of the grammatical errors and the reason for the correction.")
    is_correct: bool = Field(description="Whether the user's sentence was correct or not.")

In [ ]:
GRAMMAR_CORRECTION_SYSTEM_TEMPLATE = """
You are a helpful assistant that corrects grammar in English sentences written by users in casual or conversational contexts.
Only fix clear grammar mistakes that might confuse the listener or sound unnatural in everyday conversation.
Ignore minor issues like informal punctuation, casual phrasing, or relaxed capitalization, as long as the meaning is clear.
If the sentence is already fine for casual conversation, return it unchanged and leave feedback blank.
"""

GRAMMAR_CORRECTION_USER_TEMPLATE = """
User Input: {messages}
Feedback:
"""

def correction(state: State):
    correction_msgs = [
        ("system", GRAMMAR_CORRECTION_SYSTEM_TEMPLATE),
        ("user", GRAMMAR_CORRECTION_USER_TEMPLATE),
    ]
    correction_prompt = ChatPromptTemplate.from_messages(correction_msgs)
    model_with_structured_output = llm.with_structured_output(GrammarFeedback)
    response = model_with_structured_output.invoke(
        correction_prompt.format_messages(messages=state["messages"][-1])
    )
    print("\n[correction node]")
    print("  검토 문장:", state["messages"][-1].content)
    print("  교정 문장:", response.corrected_sentence)
    print("  피드백:", response.feedback)
    print("  is_correct:", response.is_correct)
    return {
        "messages": [AIMessage(content=response.corrected_sentence)],
        "corrected_sentence": response.corrected_sentence,
        "feedback": response.feedback,
        "is_correct": bool(response.is_correct),
    }

graph_builder.add_node("correction", correction)

## Step 4. 답변 노드 (respond)
교정 결과에 따라: 맞았으면 자연스러운 후속 질문(영어), 틀렸으면 한국어 설명 + 영어 후속 질문.

In [ ]:
CHAT_SYSTEM_TEMPLATE = """
You are an English assistant helping users improve their grammar and conversational skills.
If the CORRECT FLAG is True, skip feedback and just follow up with a natural question or comment IN ENGLISH.
If the CORRECT FLAG is False, first explain why the sentence was corrected in a clear and friendly way IN KOREAN.
Then follow up with a natural question or comment IN ENGLISH that encourages the user to continue.
"""

CHAT_USER_TEMPLATE = """
Corrected Sentence: {corrected_sentence}
Feedback Explanation: {feedback}
CORRECT FLAG: {is_correct}
Answer:
"""

def respond(state: State):
    corrected_sentence = state.get("corrected_sentence", "")
    feedback = state.get("feedback", "")
    is_correct = state.get("is_correct", False)

    chat_msgs = [
        ("system", CHAT_SYSTEM_TEMPLATE),
        ("user", CHAT_USER_TEMPLATE),
    ]
    chat_prompt = ChatPromptTemplate.from_messages(chat_msgs)
    response = llm.invoke(
        chat_prompt.format_messages(
            corrected_sentence=corrected_sentence, feedback=feedback, is_correct=is_correct
        )
    )
    print("\n[respond node] ->", response.content)
    return {"messages": [AIMessage(content=response.content)]}

graph_builder.add_node("respond", respond)

## Step 5. 라우팅 — 한국어면 번역, 영어면 교정
정규식으로 한글이 포함됐는지 보고 분기한다.

In [ ]:
import re
from langgraph.graph import START, END

def route_function(state: State):
    last_message = state["messages"][-1].content
    # 한글이 포함되어 있으면 번역 경로로
    if bool(re.search(r"[가-힣]", last_message)):
        return "translation"
    else:
        return "correction"

graph_builder.add_conditional_edges(
    START,
    route_function,
    {"translation": "translation", "correction": "correction"},
)

## Step 6. 나머지 엣지 연결 후 컴파일

In [ ]:
graph_builder.add_edge("translation", "correction")
graph_builder.add_edge("correction", "respond")
graph_builder.add_edge("respond", END)

from langgraph.checkpoint.memory import MemorySaver
memory = MemorySaver()
graph = graph_builder.compile(checkpointer=memory)

In [ ]:
from IPython.display import Image, display

try:
    display(Image(graph.get_graph().draw_mermaid_png()))
except Exception:
    print(graph.get_graph().draw_mermaid())

## 실행
한국어 입력(번역 경로)과 문법 오류가 있는 영어 입력(교정 경로)을 각각 넣어본다.

In [ ]:
config = {"configurable": {"thread_id": "1"}}

# 1) 한국어 입력 → 번역 → 교정 → 답변
from langchain_core.messages import HumanMessage
graph.invoke({"messages": [HumanMessage(content="나는 어제 영화를 봤어.")]}, config=config)
print("=" * 60)
# 2) 문법 오류 영어 입력 → 교정 → 답변
graph.invoke({"messages": [HumanMessage(content="I goed to school yesterday.")]}, config=config)

## 정리

- 조건부 분기로 입력 언어에 따라 경로를 나눔 (`route_function`)
- 구조화 출력(`GrammarFeedback`)으로 교정 결과를 깔끔하게 받음
- 멀티 노드(번역 → 교정 → 답변)를 엣지로 연결한 종합 그래프
- `MemorySaver` 로 멀티턴 대화 유지

여기까지가 LangGraph 입문 프로젝트다. 기초(basics) + 입문 프로젝트(projects)로 LangGraph 의 핵심 메커니즘과 실제 LLM 에이전트 패턴을 모두 다뤘다.